In [1]:
import pandas as pd
import numpy as np

# Data

In [4]:
resale = pd.read_csv("Data\\resale.csv")
resale.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [5]:
print(f"Shape: {resale.shape}")
print(resale.info())
resale.describe(include='all').T

Shape: (225127, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225127 entries, 0 to 225126
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                225127 non-null  object 
 1   town                 225127 non-null  object 
 2   flat_type            225127 non-null  object 
 3   block                225127 non-null  object 
 4   street_name          225127 non-null  object 
 5   storey_range         225127 non-null  object 
 6   floor_area_sqm       225127 non-null  float64
 7   flat_model           225127 non-null  object 
 8   lease_commence_date  225127 non-null  int64  
 9   remaining_lease      225127 non-null  object 
 10  resale_price         225127 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 18.9+ MB
None


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
month,225127,110,2024-07,3036,NaN,NaN,NaN,NaN,NaN,NaN,NaN
town,225127,26,SENGKANG,18391,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flat_type,225127,7,4 ROOM,95464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,225127,2749,2,682,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street_name,225127,577,YISHUN RING RD,3208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
storey_range,225127,17,04 TO 06,51639,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area_sqm,225127.0,NaN,NaN,NaN,96.751932,24.019493,31.0,81.0,93.0,112.0,366.7
flat_model,225127,21,Model A,80573,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lease_commence_date,225127.0,NaN,NaN,NaN,1996.463969,14.314362,1966.0,1985.0,1997.0,2012.0,2021.0
remaining_lease,225127,696,94 years 10 months,1919,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Data Check [None]

In [6]:
pd.DataFrame({
    'nunique': resale.nunique(dropna=False),
    'missing': resale.isna().sum(),
}).sort_values('nunique', ascending=False)

,nunique,missing
resale_price,4590,0
block,2749,0
remaining_lease,696,0
street_name,577,0
floor_area_sqm,187,0
month,110,0
lease_commence_date,56,0
town,26,0
flat_model,21,0
storey_range,17,0


## Data Description

In [7]:
# 'role': 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
# 'type':  'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
role_map = {
    "month" : "feature",
    "town": "feature",
    "flat_type": "metadata",
    "block": "identifier",
    "street_name": "identifier",
    "storey_range": "feature",
    "floor_area_sqm": "feature",
    "flat_model": "metadata",
    "lease_commence_date": "metadata",
    "remaining_lease": "feature",
    "resale_price": "target" 
}
type_map = {
    "month" : "num-disc",
    "town": "cat-nom",
    "flat_type": "cat-nom",
    "block": "cat-nom",
    "street_name": "cat-nom",
    "storey_range": "cat-ord",
    "floor_area_sqm": "num-disc",
    "flat_model": "cat-nom",
    "lease_commence_date": "num-disc",
    "remaining_lease": "num-disc",
    "resale_price": "num-disc" 
}

data_description = pd.DataFrame({
    "column": resale.columns,
})
data_description["role"] = data_description["column"].map(role_map).fillna("unknown")
data_description["type"] = data_description["column"].map(type_map).fillna("unknown")
data_description

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,metadata,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
7,flat_model,metadata,cat-nom
8,lease_commence_date,metadata,num-disc
9,remaining_lease,feature,num-disc


## Training & Test Dataset

### Model Dataset Descriptions

In [8]:
model_data_desc = data_description[(data_description['role'] != "metadata") & (data_description['role'] != "identifier")]
model_data_desc

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
9,remaining_lease,feature,num-disc
10,resale_price,target,num-disc


### Model Dataset

In [9]:
model_data = resale[model_data_desc["column"]].copy()
model_data.head()

,month,town,storey_range,floor_area_sqm,remaining_lease,resale_price
0,2017-01,ANG MO KIO,10 TO 12,44.0,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,01 TO 03,67.0,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,01 TO 03,67.0,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,04 TO 06,68.0,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,01 TO 03,67.0,62 years 05 months,265000.0


### Data Processing

#### Numerical Data

**Month** \
Convert to indexing for model input, index represents the order for time-series models \
index starts from 2017-01 onwards \
Example:

| Month(before) | index(after) |
| ------ | ------ |
| 2017-01 | 0 |
| 2017-02 | 1 |

In [10]:
months = pd.to_datetime(model_data['month'], format='%Y-%m')
model_data['month'] = (
    months.dt.year * 12 + months.dt.month
)

model_data['month'] -= model_data['month'].min()
model_data.head()

,month,town,storey_range,floor_area_sqm,remaining_lease,resale_price
0,0,ANG MO KIO,10 TO 12,44.0,61 years 04 months,232000.0
1,0,ANG MO KIO,01 TO 03,67.0,60 years 07 months,250000.0
2,0,ANG MO KIO,01 TO 03,67.0,62 years 05 months,262000.0
3,0,ANG MO KIO,04 TO 06,68.0,62 years 01 month,265000.0
4,0,ANG MO KIO,01 TO 03,67.0,62 years 05 months,265000.0


**storey_range** \
Replace the storey_range of HDB with the average storey instead \
e.g. 10 to 12 => 11

In [11]:
model_data['avg_storey'] = model_data['storey_range'].str.split(" TO ", expand=True).astype(int).mean(axis=1)
model_data.drop(columns=['storey_range'], inplace=True)
model_data.head()

,month,town,floor_area_sqm,remaining_lease,resale_price,avg_storey
0,0,ANG MO KIO,44.0,61 years 04 months,232000.0,11.0
1,0,ANG MO KIO,67.0,60 years 07 months,250000.0,2.0
2,0,ANG MO KIO,67.0,62 years 05 months,262000.0,2.0
3,0,ANG MO KIO,68.0,62 years 01 month,265000.0,5.0
4,0,ANG MO KIO,67.0,62 years 05 months,265000.0,2.0


**remaining_lease** \
Convert remaining_lease to number of months instead of X years X months

In [12]:
remaining_yrs = model_data['remaining_lease'].str.extract(r'(\d+)\D+(?:(\d+)\D+)?').fillna(0).astype(int)
model_data['remaining_lease'] = remaining_yrs[0] * 12 + remaining_yrs[1]
model_data.head()

,month,town,floor_area_sqm,remaining_lease,resale_price,avg_storey
0,0,ANG MO KIO,44.0,736,232000.0,11.0
1,0,ANG MO KIO,67.0,727,250000.0,2.0
2,0,ANG MO KIO,67.0,749,262000.0,2.0
3,0,ANG MO KIO,68.0,745,265000.0,5.0
4,0,ANG MO KIO,67.0,749,265000.0,2.0


**\*New\*** - **ppsm** \
Replace resale price with *ppsm* a new column to show the price per sq meter instead of resale price \
rounded to 2 d.p

In [13]:
model_data['ppsm'] = (model_data['resale_price'] / model_data['floor_area_sqm']).round(2)
model_data.drop(columns=['resale_price'], inplace=True)
model_data.head()

,month,town,floor_area_sqm,remaining_lease,avg_storey,ppsm
0,0,ANG MO KIO,44.0,736,11.0,5272.73
1,0,ANG MO KIO,67.0,727,2.0,3731.34
2,0,ANG MO KIO,67.0,749,2.0,3910.45
3,0,ANG MO KIO,68.0,745,5.0,3897.06
4,0,ANG MO KIO,67.0,749,2.0,3955.22


#### Categorical

**Town** \
Change to one-hot encoding 

In [14]:
model_data = pd.get_dummies(model_data, columns=["town"], drop_first=True)
model_data.head()

,month,floor_area_sqm,remaining_lease,avg_storey,ppsm,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,town_BUKIT PANJANG,...,town_PASIR RIS,town_PUNGGOL,town_QUEENSTOWN,town_SEMBAWANG,town_SENGKANG,town_SERANGOON,town_TAMPINES,town_TOA PAYOH,town_WOODLANDS,town_YISHUN
0,0,44.0,736,11.0,5272.73,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,67.0,727,2.0,3731.34,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,67.0,749,2.0,3910.45,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0,68.0,745,5.0,3897.06,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,67.0,749,2.0,3955.22,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# Export Data

In [ ]:
model_data.to_csv('Data\\processed.csv', index=False)